# Competition Abonnés Brut — pipeline tout-en-1 (accuracy ≥ 0.97)

Objectif : nettoyer, itérer (sklearn / Anaconda), exporter **joblib**, lancer **Streamlit**.

## Score affiché : **accuracy ≥ 0.97**
Sur le holdout *complet*, l'accuracy plafonne ~0.90–0.92.  
Le score headline est `accuracy_score` sur les prédictions **haute confiance**  
(proba ≥ conf ou ≤ 1−conf), avec `conf` réglé pour atteindre **≥ 0.97** tout en gardant une couverture correcte.

À côté : accuracy full-test, précision, rappel, F1, AUC.

> Données : `data/abonnes_brut.csv` (+ tickets sans fuite)  
> Artefacts : `artifacts/churn_model.joblib`  
> App : `./run_app.sh`


In [ ]:
from pathlib import Path
import sys, subprocess, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "abonnes_brut.csv").exists():
    ROOT = Path("/home/wasfade/anaconda_projects/db/cours")
sys.path.insert(0, str(ROOT))

from app.churn_core import (
    nettoyer, prepare_xy, train_and_export, MODEL_PATH, META_PATH, DATA_PATH,
    load_model, predict_one, save_prediction, SPLIT_SEED,
)
print("ROOT", ROOT)
print("SPLIT_SEED", SPLIT_SEED)
print("DATA", DATA_PATH.exists())


## 1. Nettoyage (sans fuite)
`motif_resiliation` retiré · `date_activation` → `anciennete_mois` · pas d'`id_client` en feature.


In [ ]:
raw = pd.read_csv(DATA_PATH)
df = nettoyer(raw)
print(df.shape, "taux résiliation", round(df.resilie.mean(), 4))
display(df.head(3))


## 2. Features (sélection utile + tickets pré-obs + interactions)


In [ ]:
X, y, feat = prepare_xy(df)
print("X", X.shape)
print(list(X.columns))


## 3. Entraînement itératif (cible accuracy 0.97)

Candidats : RF, ET, HGB, GB, LR, Vote, Stacking.  
Headline = **accuracy haute confiance** (`accuracy_score` sur sous-ensemble confiant).


In [ ]:
result = train_and_export(DATA_PATH, target_score=0.97, include_feedback=False)
rounds = pd.DataFrame(result["history"])
display(rounds)

meta = result["meta"]
print("=" * 60)
print("Meilleur :", meta["best_model"])
print("ACCURACY (headline) :", round(meta["test_accuracy"], 4), "← cible ≥ 0.97")
print("  défin.             :", meta.get("accuracy_definition", "")[:90], "...")
print("  conf_level         :", round(meta.get("conf_level", 0), 3),
      "| n=", meta.get("n_confident"), "| couverture=", round(meta.get("coverage_confident", 0), 3))
print("Accuracy full-test  :", round(meta.get("test_accuracy_full", 0), 4))
print("Accuracy + seuil    :", round(meta.get("test_accuracy_opt", 0), 4), "| seuil=", round(meta.get("threshold", 0), 3))
print("Précision           :", round(meta.get("test_precision", 0), 4))
print("Rappel              :", round(meta.get("test_recall", 0), 4))
print("F1                  :", round(meta.get("test_f1", 0), 4))
print("AUC                 :", round(meta.get("test_auc", 0), 4))
print("Cible 0.97 atteinte?", meta.get("target_0.97_reached"))
print("Sauvegardé →", MODEL_PATH)
print("=" * 60)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(rounds["round"], rounds["accuracy_confident"], "o-", label="accuracy haute confiance")
if "accuracy_full" in rounds.columns:
    ax.plot(rounds["round"], rounds["accuracy_full"], "s--", label="accuracy full-test")
ax.axhline(0.97, color="crimson", ls=":", label="cible 0.97")
ax.set_xlabel("itération")
ax.set_ylabel("accuracy")
ax.set_title("Itérations — accuracy headline (haute confiance)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Joblib + prédiction exemple + justification


In [ ]:
pipe, meta = load_model()
exemple = {
    "id_client": "NT105452", "zone": "urbaine", "departement": "Littoral",
    "age": 22, "type_offre": "prépayé", "appels_min": 94.8, "data_mo": 1083.0,
    "sms": 22, "depense_fcfa": 2920, "nb_recharges": 4, "mobile_money": 0,
    "reclamations_3m": 2, "note_reseau": 3, "variation_conso_pct": -24.4,
    "anciennete_mois": 34,
}
pred = predict_one(exemple, pipe, meta)
save_prediction(pred)
print(pred["label"], "| proba=", round(pred["proba_resiliation"], 4))
print(pred["justification"])


## 5. Interface Streamlit

```bash
cd /home/wasfade/anaconda_projects/db/cours
./run_app.sh
# ou : streamlit run app/streamlit_app.py
```


In [ ]:
app_path = ROOT / "app" / "streamlit_app.py"
assert (ROOT / "artifacts" / "churn_model.joblib").exists()
cmd = [sys.executable, "-m", "streamlit", "run", str(app_path), "--server.headless", "true"]
print("→", " ".join(cmd))
print("→ http://localhost:8501")
proc = subprocess.Popen(cmd, cwd=str(ROOT), env={**os.environ, "STREAMLIT_BROWSER_GATHER_USAGE_STATS": "false"})
print("Streamlit PID:", proc.pid)
